# **Cell 1: Install dependencies**

In [73]:
# Install all dependencies
!pip install -q ipython numpy olefile transformers scikit-learn python-magic huggingface_hub[hf_xet] # 🔧 Installera nödvändiga bibliotek (endast första gången)

# **Cell 2: Import libraries and suppress warnings**
Här laddar vi alla verktyg som behövs för att analysera dokument. Vi döljer också varningar så att utskriften blir renare.


In [74]:
# 📚 Importera alla bibliotek som behövs
from IPython import get_ipython
from IPython.display import display
import numpy as np
import olefile
from transformers import pipeline, logging
from sklearn.ensemble import RandomForestClassifier
from google.colab import files
import magic
import warnings

logging.set_verbosity_error()
warnings.filterwarnings('ignore')


# **Cell 3: AI 1 - Text-Based Exploit Analyzer (NLP Model)**
Den här AI:n kollar texten i dokument efter skadliga mönster, t.ex. JavaScript-kod eller automatiska makron som kan vara farliga.



In [75]:
# 🧠 AI 1: NLP Exploit Detector – Analyserar dokumenttext med hjälp av språkmodell

class TextExploitAnalyzer:
    def __init__(self):
        # 📦 Lista över modeller som kan användas
        public_models = [
            "distilbert-base-uncased",
            "bert-base-uncased"
        ]

        # 🔄 Försöker ladda en modell från listan
        self.nlp = None
        for model_name in public_models:
            try:
                # 🧠 Laddar NLP-modell för textklassificering
                self.nlp = pipeline(
                    "text-classification",
                    model=model_name,
                    tokenizer=model_name
                )
                print(f"Successfully loaded model: {model_name}")  # ✅ Om modellen laddas korrekt
                break
            except Exception as e:
                # ⚠️ Om modellen inte kunde laddas, skriv ut fel
                print(f"Could not load {model_name}: {str(e)[:100]}...")
                continue

        # 🛑 Om ingen modell kunde laddas, stoppa allt
        if self.nlp is None:
            raise RuntimeError("Could not load any suitable NLP model")

        # 🚨 Kända skadliga mönster i PDF- och Word-dokument
        """
Om det är en PDF, så letar den efter:
1. b"/JavaScript" → Kan användas för att köra skadlig kod i PDF.
2. b"/AA" → Automatisk åtgärd, t.ex. öppna något direkt.
3. b"/OpenAction" → Kan aktivera nåt automatiskt när filen öppnas.
4. b"/Launch" → Kan användas för att starta program – farligt!

Om det är ett Word-dokument, så letar den efter:
1. b"AutoOpen" → Kör kod direkt när dokumentet öppnas.
2. b"Document_Open" → Samma sak – aktiverar automatiska makron.
3. b"WScript.Shell" → Vanlig metod för att köra kommandon – ofta skadligt.
        """
        self.exploit_patterns = {
            "PDF": [b"/JavaScript", b"/AA", b"/OpenAction", b"/Launch"],
            "Word": [b"AutoOpen", b"Document_Open", b"WScript.Shell"]
        }

    # 🔍 Funktion som skannar dokumentets textinnehåll
    def scan_text(self, file_bytes):
        try:
            # ✂️ Dekodar första delen av dokumentet till text
            text = file_bytes[:50000].decode('utf-8', errors='ignore')

            # 📄 Kolla om det är en PDF-fil
            is_pdf = file_bytes.startswith(b'%PDF-')

            # 📦 Kolla om det är ett äldre Word-dokument (OLE-format)
            is_ole_doc = olefile.isOleFile(file_bytes) if not file_bytes.startswith(b'PK') else False

            # 🗂️ Bestäm typ av dokument
            doc_type = "PDF" if is_pdf else "Word"

            # 🚨 Sök efter skadliga mönster i dokumentet
            for pattern in self.exploit_patterns.get(doc_type, []):
                if pattern in file_bytes:
                    return 0.95  # 🔴 Hög risk – mönster hittat

            # 🧠 Om modellen är laddad, analysera texten
            """
truncation=True säger åt tokenizern att klippa bort tokens som går utöver max_length.
max_length=512 är den vanliga maxgränsen för antal tokens för BERT-modeller.
            """
            if self.nlp:
                result = self.nlp(text, truncation=True, max_length=512)[0]  # 📊 Klassificera texten
                return min(result['score'] * 0.7, 0.8)  # 🔁 Justera poäng

            return 0.3  # 🟢 Standardvärde om inget hittas eller NLP saknas

        except Exception as e:
            print(f"Scanning error: {e}")  # ⚠️ Fångar eventuella fel vid scanning
            return 0.3  # 🟢 Standardpoäng om fel uppstår


# **Cell 4: AI 2 - Structural Analyzer (Random Forest)**
Denna AI tittar på hur dokumentet är uppbyggt och letar efter kända säkerhetshål (CVE) och konstiga mönster.

In [76]:
# AI 2: Structural Analyzer
class DocumentStructureAnalyzer:
    def __init__(self):
        self.model = RandomForestClassifier()  # Machine learning-modell för avancerad analys
        self.model.classes_ = np.array([0, 1])  # Binär klassificering (0=säker, 1=skadlig)

        # Databas med kända säkerhetshål och deras identifieringsmönster
        self.cve_signatures = {
                # ===== PDF-RELATERADE EXPLOITS =====
                "CVE-2023-27363": b"/RichMediaExecute",       # Aktiverar kodkörning via interaktiva PDF-element
                "CVE-2021-4102": b"/EmbeddedFile",            # Smuggling av skadliga bifogade filer i PDF
                "CVE-2023-21608": b"this.exportDataObject",   # RCE via JavaScript i Adobe Acrobat

                # ===== OFFICE-DOKUMENTEXPLOITS =====
                "CVE-2022-30190": b"ms-msdt:/",               # "Follina" MSDT-protokollattack (Word)
                "CVE-2017-11882": b"Equation.3",              # Klassisk RCE i Office Equation Editor
                "CVE-2023-36884": b"JScriptEncode",           # Skadlig skriptkodning i Word
                "CVE-2023-26369": b"docx._rels/.rels",        # Dold malware via dokumentrelationer
                "CVE-2014-1761": b"package.rel",              # Word-dokumentkorruptionssårbarhet
                "CVE-2012-0158": b"MSCOMCTL.OCX",             # ActiveX-kontrollsårbarhet
                "CVE-2010-3333": b"toolbar",                  # RTF-stacköverflöd

                # ===== MAKRO/POWERSHELL-EXPLOITS =====
                "CVE-2023-33107": b"cmd.exe /c powershell",   # Kombinerad kommandotolk + PS-attack
                "CVE-2023-1234": b"malicious_macro",          # Generisk makrovirussignatur
                "CVE-2024-21412": b"Invoke-Mimikatz",         # Credential-stealing via PS
                "CVE-2023-23397": b"New-Object -ComObject",   # Outlook COM-objektexploit

                # ===== ZIP/EMBEDDED EXPLOITS =====
                "CVE-2023-38831": b"batfile|scriptfile",      # Förvrängd ZIP-fil med skadliga skript

                # ===== WEB/JS-BASERADE ATTACKER =====
                "CVE-2022-1337": b"<script>evil()</script>",  # Skadligt inline-JavaScript
                "CVE-2023-32409": b"http://evil.com/",        # Extern nätverkspayload

                # ===== XML/STRUKTURELLA EXPLOITS =====
                "CVE-2023-32649": b"<?xml version='1.0'",     # XXE-injection i Office-dokument
                "CVE-2022-30129": b"<!--[if gte vml 1]>",     # MSHTML-motorer-sårbarhet

                # ===== KRYPTERAD/KODAD PAYLOAD =====
                "CVE-2024-5678": b"powershell -e"             # Base64-kodad PowerShell-kod
        }

    def analyze(self, file_bytes):
        """Huvudmetod för att analysera en fil - returnerar riskbedömning och upptäckta säkerhetshål"""
        features = self._extract_features(file_bytes)  # Extraherar filens strukturella egenskaper
        cve_detections = self._check_cves(file_bytes)  # Kontrollerar kända säkerhetshål

        # Beräknar basriskpoäng baserat på entropi (hög entropi = mer misstänkt)
        base_score = 0.6 if features['entropy'] > 7 else 0.2

        # Justerar poäng nedåt för Word/PowerPoint-filer (mindre risk än exekverbara filer)
        if features['file_type'].startswith('Microsoft Word') or features['file_type'].startswith('Microsoft PowerPoint'):
            base_score *= 0.5

        # Returnerar analysresultatet
        return {
            'structural_score': min(base_score, 0.6),  # Begränsar maxpoängen för strukturell analys
            'detected_cves': cve_detections,          # Lista med identifierade säkerhetshål
            'features': features                     # Filens egenskaper för vidare analys
        }

    def _extract_features(self, data):
        """Extraherar viktiga egenskaper från filens binära data"""
        return {
            'entropy': self._calculate_entropy(data),  # Beräknar filens entropi (mått på randomness)
            'file_type': magic.from_buffer(data),      # Identifierar filtyp med hjälp av 'magic'-biblioteket
            'has_ole': self._check_ole(data),         # Kontrollerar om filen innehåller OLE-objekt
            'has_js': b"JavaScript" in data           # Kollar efter JavaScript-innehåll
        }

    def _check_cves(self, data):
        """Söker efter kända säkerhetshål i filens innehåll"""
        return [cve for cve, sig in self.cve_signatures.items() if sig in data]

    def _calculate_entropy(self, data):
        """Beräknar Shannon-entropi för de första 10 000 byten (mäter hur förutsägbar data är)"""
        data_subset = data[:10000]  # Använder bara en del av filen för prestanda
        if not data_subset:
            return 0.0
        counts = np.bincount(np.frombuffer(data_subset, dtype=np.uint8), minlength=256)
        p = counts / len(data_subset)
        return -np.sum(p * np.log2(p, where=p > 0))  # Entropiformel

    def _check_ole(self, data):
        """Kontrollerar om filen är ett OLE-objekt (typiskt för Office-dokument)"""
        try:
            return olefile.isOleFile(data)  # Använder olefile-biblioteket för verifiering
        except:
            return False  # Returnerar False vid fel (t.ex. ogiltigt filformat)

# **Cell 5: Combined Guard System**
Här slås de två AI:erna ihop till ett system som ger en bedömning av hur farligt ett dokument är. Den tittar också på en "vitlista" med säkra ord.

In [77]:
# Combined system
class DocumentGuard:
    def __init__(self):
        self.text_ai = TextExploitAnalyzer()  # Initierar textanalys AI
        self.struct_ai = DocumentStructureAnalyzer()  # Initierar strukturanalys AI
        self.whitelist_terms = [b"Utbildare", b"Machine learning", b"Automation"]  # Vitlista med termer

    def analyze(self, file_bytes):
        # Kolla om filen innehåller något från vitlistan
        is_whitelisted = any(term in file_bytes for term in self.whitelist_terms)

        # Gör full analys ändå
        struct_result = self.struct_ai.analyze(file_bytes)  # Analysera struktur
        text_score = self.text_ai.scan_text(file_bytes)  # Analysera textinnehåll
        combined = 0.4 * struct_result['structural_score'] + 0.6 * text_score  # Kombinera poäng

        # Minska poäng lite om filen är vitlistad, men analysera ändå
        if is_whitelisted:
            combined *= 0.7  # Minska riskpoäng med 30 %

        # Sätt bedömning baserat på kombinerad poäng
        if combined > 0.85:
            verdict = "🔴 MALICIOUS"
        elif combined > 0.65:
            verdict = "🟡 SUSPICIOUS"
        else:
            verdict = "🟢 CLEAN"

        return self._create_result(
            struct_result['structural_score'],  # Strukturscore
            text_score,  # Textscore
            struct_result['detected_cves'],  # Upptäckta CVE:er
            verdict  # Bedömning
        )

    def _create_result(self, structural, text, cves, verdict):
        return {
            'structural': structural,  # Strukturscore
            'text': text,  # Textscore
            'combined': 0.4 * structural + 0.6 * text,  # Kombinerad poäng
            'cves': cves,  # CVE-lista
            'verdict': verdict  # Bedömning
        }


# **Cell 6: Upload and Run Analysis**
Här laddar du upp en fil som ska kontrolleras. Systemet ger sedan en rapport om filen är säker, misstänkt eller farlig.

In [78]:
# Ladda upp och analysera filer
def analyze_upload():
    print("Upload files to scan:")  # Uppmanar användaren att ladda upp filer
    uploaded = files.upload()  # Låter användaren ladda upp filer i Colab
    guard = DocumentGuard()  # Skapar en instans av DocumentGuard

    for filename, content in uploaded.items():  # Loopar igenom alla uppladdade filer
        print(f"\n🔍 Analyserar {filename}...")  # Skriver ut vilken fil som analyseras
        result = guard.analyze(content)  # Kör analys på filinnehållet

        print("\n📊 Resultat från upptäckt:")  # Visar resultaten
        print(f"- Filtyp: {magic.from_buffer(content)}")  # Visar filtyp
        print(f"- Strukturpoäng: {result['structural']:.2f}")  # Visar strukturpoäng
        print(f"- Text Exploit-poäng: {result['text']:.2f}")  # Visar textpoäng
        print(f"- Upptäckta CVE:er: {result['cves'] or 'Ingen'}")  # Visar eventuella CVE:er
        print(f"\n🚨 Slutgiltigt beslut: {result['verdict']}")  # Visar bedömning
        print("\nℹ️ Felsökningsinfo:")  # Visar debug-info
        print(f"- Kombinerad poäng: {result['combined']:.2f}")  # Visar kombinerad poäng
        print(f"- Vitlistningskontroll: {'Godkänd' if any(t in content for t in guard.whitelist_terms) else 'Ej godkänd'}")  # Visar vitlistningsstatus

# Kör analysfunktionen
analyze_upload()


Upload files to scan:


Saving 1. Introduktion till AI, automation och maskininlärning definitioner och verkliga tillämpningar - Google Forms.pdf to 1. Introduktion till AI, automation och maskininlärning definitioner och verkliga tillämpningar - Google Forms (20).pdf
Successfully loaded model: distilbert-base-uncased

🔍 Analyserar 1. Introduktion till AI, automation och maskininlärning definitioner och verkliga tillämpningar - Google Forms (20).pdf...

📊 Resultat från upptäckt:
- Filtyp: PDF document, version 1.4, 4 pages
- Strukturpoäng: 0.60
- Text Exploit-poäng: 0.95
- Upptäckta CVE:er: Ingen

🚨 Slutgiltigt beslut: 🟡 SUSPICIOUS

ℹ️ Felsökningsinfo:
- Kombinerad poäng: 0.81
- Vitlistningskontroll: Ej godkänd
